# Pipeline de Geração da Base de Comparação (somente Lattes)

Este notebook gera um arquivo `.duckdb` para ser enviado no uploader **"Base
de Comparação"** do módulo *Comparativo entre Bases* do `app.py`. Diferente
de `analyse_organizado.ipynb` (que cruza Lattes + ORCID + Scopus e produz o
banco institucional de 11 tabelas, `pesquisadores_teste.duckdb`), a base
enviada para comparação tipicamente vem de **outra instituição/programa**,
para a qual só temos os currículos Lattes brutos (sem ORCID nem Scopus
vinculados) — por isso este pipeline é uma versão enxuta, restrita apenas à
extração Lattes.

## O que este notebook produz

`carregar_base_comparacao()` em `app.py` só exige 4 tabelas com uma
arquitetura mínima:

- `tb_professores` (`id_lattes`, `nome_completo`, ...)
- `tb_artigo_periodico` (com `ano_pub`, `maior_percentil`, `fontes`, ...)
- `tb_artigo_conferencia` (com `ano`, `estrato`, `fontes`, ...)
- `tb_orientacoes` (com `ano_inicio`, `ano_conclusao`, `status`, `nivel`)

Este notebook cria exatamente essas 4 tabelas, com **o mesmo schema de
colunas** usado em `tb_professores`/`tb_artigo_periodico`/
`tb_artigo_conferencia`/`tb_orientacoes` de `analyse_organizado.ipynb` — para
que a coluna `fontes` (usada pelo filtro global "Apenas cadastradas no
Lattes" do `app.py`) e as colunas de classificação (`maior_percentil`,
`estrato`) usadas na aba "Avaliação Quadrienal" do comparativo também
funcionem na Base B. Como só existe uma fonte aqui, `fontes` é sempre
`'LATTES'`.

**Não são criadas**: `tb_alunos` e as 6 tabelas por fonte
(`tb_artigo_*_lattes/orcid/scopus`) do banco institucional — nenhuma delas é
lida pela página "Comparativo entre Bases", então ficaram fora de escopo
para manter este notebook simples e autocontido.

## Estrutura deste notebook

1. Configuração e imports
2. Extração dos JSONs brutos (Lattes) — pessoas, orientações, artigos de periódico e trabalhos de congresso
3. Tratamento de `df_pessoas`
4. Tratamento de `df_orientacoes`
5. Tratamento e reshape de `df_bib_artigos`/`df_bib_trab_congresso` para o schema final
6. Deduplicação das publicações (por professor, sempre exata — nunca fuzzy)
7. Cruzamento de periódicos com a base de percentil Scopus
8. Cruzamento de trabalhos de congresso com a base de eventos classificados
9. Persistência no DuckDB (4 tabelas)

## 1. Configuração e Imports

In [ ]:
import json
import re
import glob
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb
from rapidfuzz import fuzz

# Caminho onde estão os currículos Lattes (em JSON) dos professores da
# instituição/programa usado como Base de Comparação. Aponta, por padrão,
# para o dump já existente do PPGI-UFRJ -- troque para a pasta correta antes
# de rodar se a comparação for com outra instituição.
CAMINHO_JSONS_PROFESSORES = 'outros_programas/puc/puc_scrapping/teste-01/json/*.json'

# Planilha de percentis Scopus por periódico (mesma referência estática
# usada em analyse_organizado.ipynb -- não depende de credenciais Scopus).
ARQUIVO_PERCENTIL_SCOPUS = 'periodicos_percentil.xlsx'

# Base de eventos/conferências já classificados por estrato (idem, sem
# depender de nenhuma API externa).
ARQUIVO_EVENTOS_CLASSIFICADOS = 'eventos_classificados_dois_idiomas.csv'

# Arquivo DuckDB de destino: o arquivo que deve ser enviado no uploader
# "Base de Comparação" (barra lateral, módulo "Comparativo entre Bases").
ARQUIVO_DUCKDB_DESTINO = 'pesquisadores_comparacao.duckdb'

print("OK: configuração e imports carregados.")

OK: configuração e imports carregados.


## 2. Extração dos JSONs Brutos (Lattes) e Consolidação em DataFrames

Mesma extração "achatada" (*flatten*) de `analyse_organizado.ipynb`, mas
restrita aos quatro blocos realmente usados a partir daqui:
`informacoes_pessoais`, `orientacoes` e, dentro de `producao_bibliografica`,
`artigos_periodicos` e `trabalhos_completos_congressos`. Os demais blocos do
JSON (bancas, eventos, prêmios, projetos, produção técnica, patentes) não
alimentam nenhuma tabela consumida pelo comparativo, então são ignorados
aqui.

In [2]:
print("Localizando arquivos JSON de professores...")
caminhos_arquivos = glob.glob(CAMINHO_JSONS_PROFESSORES)

if not caminhos_arquivos:
    raise FileNotFoundError(
        f"Nenhum arquivo JSON encontrado em '{CAMINHO_JSONS_PROFESSORES}'. "
        "Verifique se o caminho está correto antes de continuar."
    )

print(f"{len(caminhos_arquivos)} arquivo(s) encontrado(s). Iniciando a extração...")

Localizando arquivos JSON de professores...
31 arquivo(s) encontrado(s). Iniciando a extração...


In [3]:
lista_pessoas = []
lista_orientacoes = []
lista_bib_artigos = []
lista_bib_trabalhos_congresso = []

for arquivo in caminhos_arquivos:
    with open(arquivo, 'r', encoding='utf-8') as f:
        dados = json.load(f)

        id_lattes = dados.get('informacoes_pessoais', {}).get('id_lattes')
        if not id_lattes:
            # Sem id_lattes não há como vincular nenhum registro a um professor;
            # o arquivo é descartado.
            continue

        # --- INFORMAÇÕES PESSOAIS ---
        df_pessoa = pd.json_normalize(dados['informacoes_pessoais'])
        lista_pessoas.append(df_pessoa)

        # --- ORIENTAÇÕES (estrutura: {status: {nivel: [itens]}}) ---
        if 'orientacoes' in dados:
            for status, dicionario_niveis in dados['orientacoes'].items():
                for nivel, itens in dicionario_niveis.items():
                    if itens:
                        df_temp = pd.DataFrame(itens)
                        df_temp['id_lattes'] = id_lattes
                        df_temp['status'] = status
                        df_temp['nivel'] = nivel
                        lista_orientacoes.append(df_temp)

        # --- PRODUÇÃO BIBLIOGRÁFICA (estrutura: {chave: [itens]}) ---
        prod_bib = dados.get('producao_bibliografica', {})

        def add_to_list(chave, lista_destino):
            """Extrai uma chave de produção bibliográfica e empilha na lista destino."""
            itens = prod_bib.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list('artigos_periodicos', lista_bib_artigos)
        add_to_list('trabalhos_completos_congressos', lista_bib_trabalhos_congresso)

print("Extração e achatamento concluídos para todos os arquivos.")

Extração e achatamento concluídos para todos os arquivos.


In [4]:
def consolidar(lista):
    """Concatena uma lista de DataFrames parciais; retorna DataFrame vazio se a lista estiver vazia."""
    return pd.concat(lista, ignore_index=True) if lista else pd.DataFrame()

df_pessoas = consolidar(lista_pessoas)
df_orientacoes = consolidar(lista_orientacoes)
df_bib_artigos = consolidar(lista_bib_artigos)
df_bib_trab_congresso = consolidar(lista_bib_trabalhos_congresso)

print("DataFrames consolidados. Resumo de volumes:")
print(f"  Professores (df_pessoas):              {len(df_pessoas)}")
print(f"  Orientações (df_orientacoes):           {len(df_orientacoes)}")
print(f"  Artigos de periódico (df_bib_artigos):  {len(df_bib_artigos)}")
print(f"  Trabalhos de congresso (df_bib_trab_congresso): {len(df_bib_trab_congresso)}")

DataFrames consolidados. Resumo de volumes:
  Professores (df_pessoas):              31
  Orientações (df_orientacoes):           2409
  Artigos de periódico (df_bib_artigos):  1604
  Trabalhos de congresso (df_bib_trab_congresso): 3119


## 3. Tratamento de `df_pessoas` (Informações Pessoais)

Limpeza padrão de cadastro: strings vazias→nulo, datas, remoção de marcador
"*" no rótulo, tipagem da chave primária e do texto de resumo -- idêntico a
`analyse_organizado.ipynb`. `orcid_id`, `scopus_author_id` e `data_ingresso`
ficam sempre nulos aqui: esta base é Lattes-only por definição e não depende
de `lista_pessoas.csv` (que é específico da base institucional), mas as três
colunas continuam existindo em `tb_professores` para manter a mesma
arquitetura de tabela que `app.py` espera.

In [5]:
# 3.1 Substitui strings vazias ou só com espaços por NaN (nulo real)
df_pessoas.replace(r'^\s*$', np.nan, regex=True, inplace=True)

# 3.2 Converte a data de atualização do CV ('15/10/2025') para datetime.
#     errors='coerce' faz datas inválidas virarem nulo em vez de quebrar o script.
if 'atualizacao_cv' in df_pessoas.columns:
    df_pessoas['atualizacao_cv'] = pd.to_datetime(
        df_pessoas['atualizacao_cv'],
        format='%d/%m/%Y',
        errors='coerce'
    )

# 3.3 Limpeza do campo 'rotulo': remove o asterisco e espaços, e transforma
#     o texto literal "Sem rótulo" em nulo verdadeiro.
if 'rotulo' in df_pessoas.columns:
    df_pessoas['rotulo'] = df_pessoas['rotulo'].str.replace('*', '', regex=False).str.strip()
    df_pessoas['rotulo'] = df_pessoas['rotulo'].replace('Sem rótulo', np.nan)

# 3.4 Garante que a chave primária (id_lattes) seja sempre string,
#     evitando inconsistências de tipo em merges/joins posteriores.
df_pessoas['id_lattes'] = df_pessoas['id_lattes'].astype(str)

# 3.5 Remove espaços/quebras de linha nas bordas do texto de resumo do CV.
if 'texto_resumo' in df_pessoas.columns:
    df_pessoas['texto_resumo'] = df_pessoas['texto_resumo'].str.strip()

# 3.6 Base Lattes-only: sem lista_pessoas.csv, sem ORCID/Scopus vinculados.
df_pessoas['orcid_id'] = pd.NA
df_pessoas['scopus_author_id'] = pd.NA
df_pessoas['data_ingresso'] = pd.NA

print("Tratamento de df_pessoas concluído.")
df_pessoas.info()

Tratamento de df_pessoas concluído.
<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id_lattes              31 non-null     str           
 1   nome_completo          31 non-null     str           
 2   nome_citacoes          31 non-null     str           
 3   sexo                   31 non-null     str           
 4   rotulo                 0 non-null      str           
 5   periodo                0 non-null      str           
 6   bolsa_produtividade    17 non-null     str           
 7   endereco_profissional  31 non-null     str           
 8   atualizacao_cv         31 non-null     datetime64[us]
 9   url                    31 non-null     str           
 10  texto_resumo           31 non-null     str           
 11  orcid_id               0 non-null      object        
 12  scopus_author_id       0 non-null      ob

## 4. Tratamento de `df_orientacoes`

**Atenção à ordem**: a coluna original `titulo` é renomeada para
`titulo_trabalho` *antes* da limpeza de texto em lote, porque a limpeza já
referencia o nome novo (`titulo_trabalho`).

> **Nota de atenção (herdada de `analyse_organizado.ipynb`):** a Seção 9
> insere `df_orientacoes` no banco esperando uma coluna `ano_inicio`. Essa
> coluna nunca é criada nem renomeada em nenhuma etapa de tratamento -- ela
> só existe se o JSON bruto de orientações já trouxer um campo chamado
> literalmente `ano_inicio`. Se o JSON de origem usar outro nome, adicione
> aqui um `rename` antes de chegar à Seção 9, ou a inserção no banco falhará.

In [ ]:
# 4.1 Renomeia 'titulo' para 'titulo_trabalho' (nome mais descritivo,
#     usado pela limpeza de texto na sequência e pelo schema do banco).
df_orientacoes = df_orientacoes.rename(columns={'titulo': 'titulo_trabalho'})

print("Tratamento da tabela de orientações...")

if not df_orientacoes.empty:
    # 4.2 Limpeza de texto: remove espaços duplos/quebras de linha escondidas e
    #     preenche vazios com um rótulo explícito em vez de deixá-los como string vazia.
    colunas_texto = ['titulo_trabalho', 'orientando', 'tipo_trabalho', 'instituicao', 'curso']
    for col in colunas_texto:
        df_orientacoes[col] = df_orientacoes[col].astype(str).str.strip()
        df_orientacoes[col] = df_orientacoes[col].replace(
            {'': 'Não informado', 'nan': 'Não informado', 'None': 'Não informado'}
        )

    # 4.3 Conversão segura do ano de conclusão (float -> Int64, que aceita nulos).
    df_orientacoes['ano_conclusao'] = df_orientacoes['ano_conclusao'].astype('Int64')

    # 4.4 Padroniza os valores de 'nivel' para rótulos amigáveis (usados em gráficos).
    mapeamento_nivel = {
        'mestrado': 'Mestrado',
        'doutorado': 'Doutorado',
        'tcc': 'TCC',
        'iniciacao_cientifica': 'Iniciação Científica',
        'pos_doutorado': 'Pós-Doutorado',
        'especializacao': 'Especialização',
        'outros': 'Outros'
    }
    df_orientacoes['nivel'] = df_orientacoes['nivel'].map(mapeamento_nivel).fillna(df_orientacoes['nivel'])

    # 4.5 Padroniza os valores de 'status' para rótulos amigáveis.
    mapeamento_status = {
        'concluidas': 'Concluída',
        'em_andamento': 'Em Andamento'
    }
    df_orientacoes['status'] = df_orientacoes['status'].map(mapeamento_status).fillna(df_orientacoes['status'])

print("Tratamento concluído. Amostra dos dados tratados:")
if not df_orientacoes.empty:
    display(df_orientacoes[['orientando', 'nivel', 'status', 'ano_conclusao']].head())
else:
    print("(nenhuma orientação encontrada nesta base -- tabela vazia)")


## 5. Tratamento de `df_bib_artigos`/`df_bib_trab_congresso` e Reshape

Limpeza básica (nulos reais, maiúsculas no nome do periódico/evento, tipagem
de ano, texto sem espaços ocultos) e, em seguida, *reshape* para o schema
final -- as mesmas colunas de `tb_artigo_periodico`/`tb_artigo_conferencia`
usadas por `analyse_organizado.ipynb`, com os campos que dependem de
cruzamento (`maior_percentil`, `estrato`, `match_adequado` etc.) começando
nulos e uma coluna `fonte='LATTES'` fixa (só existe uma fonte aqui).

In [ ]:
print("Aplicando tratamentos na tabela 'df_bib_artigos'...")

if not df_bib_artigos.empty:
    # 1. Strings vazias/só espaços -> nulo real
    df_bib_artigos.replace(r'^\s*$', np.nan, regex=True, inplace=True)

    # 2. Nome da revista em maiúsculas e sem espaços nas bordas
    #    (necessário para o cruzamento exato com a base Scopus na Seção 7)
    if 'revista' in df_bib_artigos.columns:
        df_bib_artigos['revista'] = df_bib_artigos['revista'].str.upper().str.strip()

    # 3. Ano como inteiro com suporte a nulo (Int64); valores inválidos -> nulo
    if 'ano' in df_bib_artigos.columns:
        df_bib_artigos['ano'] = pd.to_numeric(df_bib_artigos['ano'], errors='coerce').astype('Int64')

    # 4. Remove espaços ocultos nas colunas de texto livre
    colunas_texto = ['titulo', 'doi', 'issn', 'volume', 'numero', 'paginas']
    for col in colunas_texto:
        if col in df_bib_artigos.columns:
            df_bib_artigos[col] = df_bib_artigos[col].str.strip()

    # 5. Garante tipagem string na chave primária
    df_bib_artigos['id_lattes'] = df_bib_artigos['id_lattes'].astype(str)

print("Tratamento de 'df_bib_artigos' concluído.")
if not df_bib_artigos.empty:
    display(df_bib_artigos[['ano', 'revista', 'doi', 'issn']].head())
else:
    print("(nenhum artigo de periódico encontrado nesta base -- tabela vazia)")

In [ ]:
print("Aplicando tratamentos na tabela 'df_bib_trab_congresso'...")

if not df_bib_trab_congresso.empty:
    # 1. Strings vazias/só espaços -> nulo real
    df_bib_trab_congresso.replace(r'^\s*$', np.nan, regex=True, inplace=True)

    # 2. Nome do evento em maiúsculas e sem espaços nas bordas
    #    (necessário para o cruzamento com a base de eventos na Seção 8)
    if 'evento' in df_bib_trab_congresso.columns:
        df_bib_trab_congresso['evento'] = df_bib_trab_congresso['evento'].str.upper().str.strip()

    # 3. Ano como inteiro com suporte a nulo (Int64)
    if 'ano' in df_bib_trab_congresso.columns:
        df_bib_trab_congresso['ano'] = pd.to_numeric(df_bib_trab_congresso['ano'], errors='coerce').astype('Int64')

    # 4. Remove espaços ocultos nas colunas de texto livre
    colunas_texto = ['titulo', 'doi', 'isbn', 'paginas']
    for col in colunas_texto:
        if col in df_bib_trab_congresso.columns:
            df_bib_trab_congresso[col] = df_bib_trab_congresso[col].str.strip()

    # 5. Padronização da coluna 'autores': separador único (vírgula),
    #    sem espaços duplicados, e em maiúsculas.
    if 'autores' in df_bib_trab_congresso.columns:
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.strip()
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.replace(';', ',', regex=False)
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.replace(r'\s+', ' ', regex=True)
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.upper()

    # 6. Garante tipagem string na chave primária
    df_bib_trab_congresso['id_lattes'] = df_bib_trab_congresso['id_lattes'].astype(str)

print("Tratamento de 'df_bib_trab_congresso' concluído.")
if not df_bib_trab_congresso.empty:
    display(df_bib_trab_congresso[['ano', 'evento', 'autores']].head())
else:
    print("(nenhum trabalho de congresso encontrado nesta base -- tabela vazia)")

In [9]:
COLUNAS_PERIODICO = [
    'id_lattes', 'titulo_artigo', 'titulo_revista_lattes', 'ano_pub', 'doi',
    'autores', 'match_adequado', 'coautoria_aluno', 'id_scopus',
    'titulo_revista_scopus', 'maior_percentil', 'codigo_area_maior_percentil',
    'area_maior_percentil', 'issn', 'computation_area', 'fonte',
]

COLUNAS_CONGRESSO = [
    'id_lattes', 'titulo_artigo', 'ano', 'doi', 'autores',
    'titulo_evento_lattes', 'paginas', 'sigla_evento_google',
    'titulo_evento_google', 'estrato', 'tipo_match', 'coautoria_aluno', 'fonte',
]


def montar_df_vazio(colunas):
    """Cria um DataFrame vazio já com as colunas do schema final, evitando
    KeyError mais adiante quando a extração não retorna nenhum registro."""
    return pd.DataFrame(columns=colunas)


print("Reorganizando 'df_bib_artigos' para o schema final (fonte LATTES)...")
if not df_bib_artigos.empty:
    df_artigos_periodico_lattes = pd.DataFrame({
        'id_lattes': df_bib_artigos['id_lattes'],
        'titulo_artigo': df_bib_artigos.get('titulo'),
        'titulo_revista_lattes': df_bib_artigos.get('revista'),
        'ano_pub': df_bib_artigos.get('ano'),
        'doi': df_bib_artigos.get('doi'),
        'autores': df_bib_artigos.get('autores'),
        'match_adequado': pd.NA,
        'coautoria_aluno': pd.NA,
        'id_scopus': pd.NA,
        'titulo_revista_scopus': pd.NA,
        'maior_percentil': pd.NA,
        'codigo_area_maior_percentil': pd.NA,
        'area_maior_percentil': pd.NA,
        'issn': df_bib_artigos.get('issn'),
        'computation_area': pd.NA,
        'fonte': 'LATTES',
    })[COLUNAS_PERIODICO]
else:
    df_artigos_periodico_lattes = montar_df_vazio(COLUNAS_PERIODICO)

print("Reorganizando 'df_bib_trab_congresso' para o schema final (fonte LATTES)...")
if not df_bib_trab_congresso.empty:
    df_artigos_congresso_lattes = pd.DataFrame({
        'id_lattes': df_bib_trab_congresso['id_lattes'],
        'titulo_artigo': df_bib_trab_congresso.get('titulo'),
        'ano': df_bib_trab_congresso.get('ano'),
        'doi': df_bib_trab_congresso.get('doi'),
        'autores': df_bib_trab_congresso.get('autores'),
        'titulo_evento_lattes': df_bib_trab_congresso.get('evento'),
        'paginas': df_bib_trab_congresso.get('paginas'),
        'sigla_evento_google': pd.NA,
        'titulo_evento_google': pd.NA,
        'estrato': pd.NA,
        'tipo_match': pd.NA,
        'coautoria_aluno': pd.NA,
        'fonte': 'LATTES',
    })[COLUNAS_CONGRESSO]
else:
    df_artigos_congresso_lattes = montar_df_vazio(COLUNAS_CONGRESSO)

print(f"df_artigos_periodico_lattes: {len(df_artigos_periodico_lattes)} linhas")
print(f"df_artigos_congresso_lattes: {len(df_artigos_congresso_lattes)} linhas")
display(df_artigos_periodico_lattes.head())

Reorganizando 'df_bib_artigos' para o schema final (fonte LATTES)...
Reorganizando 'df_bib_trab_congresso' para o schema final (fonte LATTES)...
df_artigos_periodico_lattes: 1604 linhas
df_artigos_congresso_lattes: 3119 linhas


,id_lattes,titulo_artigo,titulo_revista_lattes,ano_pub,doi,autores,match_adequado,coautoria_aluno,id_scopus,titulo_revista_scopus,maior_percentil,codigo_area_maior_percentil,area_maior_percentil,issn,computation_area,fonte
0,0211300683784278,On the (In)Dependence of the Peano Axioms for ...,HISTORY AND PHILOSOPHY OF LOGIC,2021,http://dx.doi.org/10.1080/01445340.2021.1971005,"CERIOLI, MÁRCIA R.; NOBREGA, HUGO ; SILVEIRA, ...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1464-5149,<NA>,LATTES
1,0211300683784278,Short proofs on the structure of general parti...,DISCRETE APPLIED MATHEMATICS,2021,http://dx.doi.org/10.1016/j.dam.2020.09.007,"CERIOLI, MÁRCIA R.; MARTINS, TAÍSA",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0166-218X,<NA>,LATTES
2,0211300683784278,Transversals of longest paths,DISCRETE MATHEMATICS,2020,http://dx.doi.org/10.1016/j.disc.2019.111717,"CERIOLI, MÁRCIA R.; FERNANDES, CRISTINA G. ; G...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0012-365X,<NA>,LATTES
3,0211300683784278,Intersection of longest paths in graph classes,DISCRETE APPLIED MATHEMATICS,2020,http://dx.doi.org/10.1016/j.dam.2019.03.022,"CERIOLI, MÁRCIA R.; LIMA, PALOMA T.",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0166-218X,<NA>,LATTES
4,0211300683784278,On Edge-magic Labelings of Forests,ELECTRONIC NOTES IN THEORETICAL COMPUTER SCIENCE,2019,http://dx.doi.org/10.1016/j.entcs.2019.08.027,"CERIOLI, M. R.; FERNANDES, C. G. ; LEE, O. ; L...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1571-0661,<NA>,LATTES


## 6. Deduplicação das Publicações (por professor, sempre exata)

Mesmo com uma única fonte, o próprio Lattes pode listar a mesma publicação
duas vezes (erro de cadastro do professor), então a deduplicação continua
necessária. Regras idênticas a `analyse_organizado.ipynb`, Seção 8:

- A deduplicação é feita **dentro da lista de cada professor**, isoladamente
  -- nunca entre professores, para não fundir coautoria de dois professores
  do quadro em uma única linha.
- Duas linhas do mesmo professor são a mesma publicação apenas se têm o
  mesmo **DOI normalizado** ou o mesmo **título normalizado** -- comparação
  **sempre exata**, nunca por similaridade/fuzzy.

`calcular_chave_dedup`, `deduplicar_bruto_por_fonte` e `unificar_com_dedup`
são reaproveitadas sem alteração: mesmo com uma fonte só, elas continuam
corretas (o resultado de `unificar_com_dedup` grava `fontes='LATTES'` em
toda linha).

In [10]:
def normalizar_doi(doi):
    """Reduz um DOI à sua forma canônica para comparação EXATA: minúsculo,
    sem espaços, sem prefixo textual ("doi:") nem de URL
    (https://doi.org/, http://dx.doi.org/) e sem pontuação solta na borda
    (várias fontes trazem o DOI seguido de ponto final). Retorna <NA> quando
    não sobra nada de útil -- nesse caso a linha simplesmente não participa
    do casamento por DOI."""
    if pd.isna(doi):
        return pd.NA
    texto = str(doi).strip().lower()
    texto = re.sub(r'^doi\s*:\s*', '', texto)
    texto = re.sub(r'^https?://(dx\.)?doi\.org/', '', texto)
    texto = texto.strip(' .,;')
    if texto in ('', 'nan', 'none', '<na>'):
        return pd.NA
    return texto


def normalizar_titulo_dedup(titulo):
    """Reduz um título à sua forma canônica para comparação EXATA:
    maiúsculas, sem acentuação, sem pontuação e com espaços colapsados.
    Retorna '' quando não há título -- nesse caso a linha não participa do
    casamento por título."""
    if pd.isna(titulo):
        return ''
    texto = str(titulo).upper().strip()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('ascii')
    texto = re.sub(r'[^A-Z0-9 ]+', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()

In [ ]:
def calcular_chave_dedup(df):
    """Atribui a cada linha a chave da publicação a que ela pertence.

    Regra fundamental: a deduplicação acontece dentro da lista de cada
    professor, nunca entre professores. As linhas são agrupadas por
    id_lattes e o casamento (por DOI ou por título) só é procurado dentro
    do grupo; a chave gerada carrega o id_lattes como prefixo, então duas
    linhas de professores diferentes nunca podem cair na mesma chave.

    Dentro do grupo de um professor, duas linhas são a mesma publicação se:
      - têm o mesmo DOI normalizado (comparação exata), OU
      - têm o mesmo título normalizado (comparação exata).
    Nenhum limiar de similaridade é usado em nenhum dos dois critérios.

    A chave final de cada grupo é id_lattes|DOI:<doi> quando alguma linha
    do grupo tem DOI (é o identificador mais forte) e id_lattes|TIT:<titulo>
    caso contrário. Linhas sem DOI e sem título não casam com nada e recebem
    uma chave própria (id_lattes|LINHA:<n>), para não serem fundidas entre si.
    """
    total = len(df)
    if total == 0:
        return pd.Series([], dtype='object', index=df.index)

    dois = df['doi'].map(normalizar_doi).tolist()
    titulos = df['titulo_artigo'].map(normalizar_titulo_dedup).tolist()
    professores = df['id_lattes'].astype(str).tolist()

    posicoes_por_professor = {}
    for posicao in range(total):
        posicoes_por_professor.setdefault(professores[posicao], []).append(posicao)

    chaves = [None] * total

    for professor, posicoes in posicoes_por_professor.items():
        pai = {posicao: posicao for posicao in posicoes}

        def encontrar(x):
            while pai[x] != x:
                pai[x] = pai[pai[x]]
                x = pai[x]
            return x

        def unir(a, b):
            raiz_a, raiz_b = encontrar(a), encontrar(b)
            if raiz_a != raiz_b:
                pai[raiz_b] = raiz_a

        primeira_com_doi = {}
        primeira_com_titulo = {}
        for posicao in posicoes:
            doi = dois[posicao]
            if not pd.isna(doi):
                unir(primeira_com_doi.setdefault(doi, posicao), posicao)
            titulo = titulos[posicao]
            if titulo:
                unir(primeira_com_titulo.setdefault(titulo, posicao), posicao)

        doi_do_grupo = {}
        for posicao in posicoes:
            doi = dois[posicao]
            if pd.isna(doi):
                continue
            raiz = encontrar(posicao)
            if raiz not in doi_do_grupo or doi < doi_do_grupo[raiz]:
                doi_do_grupo[raiz] = doi

        for posicao in posicoes:
            raiz = encontrar(posicao)
            if raiz in doi_do_grupo:
                identificador = f'DOI:{doi_do_grupo[raiz]}'
            elif titulos[raiz]:
                identificador = f'TIT:{titulos[raiz]}'
            else:
                identificador = f'LINHA:{raiz}'
            chaves[posicao] = f'{professor}|{identificador}'

    return pd.Series(chaves, index=df.index, dtype='object')


ORDEM_PRIORIDADE_FONTE = {'LATTES': 0}


def deduplicar_bruto_por_fonte(df, chave='chave_dedup'):
    """Garante uma única linha por (publicação, fonte). Sem isso, uma
    repetição vinda da própria extração (o mesmo artigo listado duas vezes
    no currículo Lattes) sobreviveria até as tabelas usadas nos relatórios."""
    if df.empty:
        return df
    df_ordenado = df.copy()
    df_ordenado['_prioridade'] = df_ordenado['fonte'].map(ORDEM_PRIORIDADE_FONTE).fillna(9)
    df_ordenado = df_ordenado.sort_values('_prioridade', kind='stable')
    return (
        df_ordenado.drop_duplicates(subset=['fonte', chave], keep='first')
        .drop(columns=['_prioridade'])
        .reset_index(drop=True)
    )


def unificar_com_dedup(df, chave='chave_dedup'):
    """Reduz a base bruta (uma linha por publicação por fonte) à base final
    (uma linha por publicação de cada professor). Cada coluna recebe o
    primeiro valor não nulo encontrado no grupo, e a coluna fontes registra
    todas as bases em que aquela publicação foi encontrada (aqui, sempre
    'LATTES', já que só há uma fonte)."""
    if df.empty:
        return pd.DataFrame(columns=[c for c in df.columns if c != 'fonte'] + ['fontes'])

    df_ordenado = df.copy()
    df_ordenado['_prioridade'] = df_ordenado['fonte'].map(ORDEM_PRIORIDADE_FONTE).fillna(9)
    df_ordenado = df_ordenado.sort_values('_prioridade', kind='stable')

    colunas_dado = [c for c in df.columns if c not in (chave, 'fonte', '_prioridade')]
    linhas_unicas = df_ordenado.drop_duplicates(subset=[chave], keep='first')[[chave] + colunas_dado].copy()

    for coluna in colunas_dado:
        if not linhas_unicas[coluna].isna().any():
            continue
        preenchimento = (
            df_ordenado.dropna(subset=[coluna])
            .drop_duplicates(subset=[chave], keep='first')[[chave, coluna]]
            .rename(columns={coluna: '_preenchimento'})
        )
        linhas_unicas = linhas_unicas.merge(preenchimento, on=chave, how='left')
        linhas_unicas[coluna] = linhas_unicas[coluna].fillna(linhas_unicas['_preenchimento'])
        linhas_unicas = linhas_unicas.drop(columns=['_preenchimento'])

    fontes_por_publicacao = (
        df.groupby(chave)['fonte']
        .apply(lambda serie: ','.join(sorted(set(serie.dropna()))))
        .rename('fontes')
        .reset_index()
    )
    return linhas_unicas.merge(fontes_por_publicacao, on=chave, how='left').reset_index(drop=True)


def auditar_duplicatas(df_unificado, df_bruto, rotulo):
    """Confere (sem alterar nada) se o resultado bate com os requisitos:
    nenhum professor com DOI repetido, nenhum professor com título repetido,
    nenhuma chave repetida na base unificada, nenhuma linha repetida dentro
    da mesma fonte na base bruta e nenhuma chave compartilhada por
    professores diferentes."""
    if df_unificado.empty:
        print(f"  OK ({rotulo}): nenhuma publicação nesta base -- nada para auditar.")
        return

    doi_norm = df_unificado['doi'].map(normalizar_doi)
    titulo_norm = df_unificado['titulo_artigo'].map(normalizar_titulo_dedup)
    professor = df_unificado['id_lattes'].astype(str)

    dup_doi = pd.concat([professor, doi_norm], axis=1).duplicated(keep=False) & doi_norm.notna()
    dup_titulo = (professor + '|' + titulo_norm).duplicated(keep=False) & (titulo_norm != '')
    dup_chave = df_unificado['chave_dedup'].duplicated(keep=False)
    dup_fonte = df_bruto.duplicated(subset=['fonte', 'chave_dedup'], keep=False)
    chaves_multiprofessor = (
        df_unificado.groupby('chave_dedup')['id_lattes'].nunique().gt(1).sum()
    )

    total_problemas = (
        int(dup_doi.sum()) + int(dup_titulo.sum()) + int(dup_chave.sum())
        + int(dup_fonte.sum()) + int(chaves_multiprofessor)
    )
    if total_problemas == 0:
        print(f"  OK ({rotulo}): nenhum professor tem artigo repetido; nenhuma chave cruzou professores.")
        return

    if dup_doi.any():
        print(f"  AVISO ({rotulo}): {int(dup_doi.sum())} linha(s) com o mesmo DOI para o mesmo professor.")
    if dup_titulo.any():
        print(f"  AVISO ({rotulo}): {int(dup_titulo.sum())} linha(s) com o mesmo título para o mesmo professor.")
    if dup_chave.any():
        print(f"  AVISO ({rotulo}): {int(dup_chave.sum())} linha(s) com chave_dedup repetida na base unificada.")
    if dup_fonte.any():
        print(f"  AVISO ({rotulo}): {int(dup_fonte.sum())} linha(s) repetida(s) dentro da mesma fonte (base bruta).")
    if chaves_multiprofessor:
        print(f"  AVISO ({rotulo}): {int(chaves_multiprofessor)} chave(s) compartilhada(s) por professores diferentes.")


print("Calculando a chave de deduplicação (por professor, via DOI ou título)...")
df_artigos_periodico_lattes['chave_dedup'] = calcular_chave_dedup(df_artigos_periodico_lattes)
df_artigos_congresso_lattes['chave_dedup'] = calcular_chave_dedup(df_artigos_congresso_lattes)

print("Removendo repetições dentro da própria extração Lattes...")
df_periodicos_bruto = deduplicar_bruto_por_fonte(df_artigos_periodico_lattes)
df_congressos_bruto = deduplicar_bruto_por_fonte(df_artigos_congresso_lattes)

print("Unificando (uma linha por publicação de cada professor)...")
df_periodicos_unificado = unificar_com_dedup(df_periodicos_bruto)
df_congressos_unificado = unificar_com_dedup(df_congressos_bruto)

print("\nConferindo se sobrou alguma duplicata:")
auditar_duplicatas(df_periodicos_unificado, df_periodicos_bruto, 'periódicos')
auditar_duplicatas(df_congressos_unificado, df_congressos_bruto, 'congressos')

print(f"\nPeriódicos: {len(df_periodicos_unificado)} publicações únicas")
print(f"Congressos: {len(df_congressos_unificado)} publicações únicas")
display(df_periodicos_unificado[['id_lattes', 'titulo_artigo', 'doi', 'fontes']].head())

## 7. Cruzamento de Periódicos com a Base de Percentil Scopus

Mesma lógica em camadas de `analyse_organizado.ipynb` (Seção 9): match exato
por nome do periódico, match exato por ISSN e, só para o que sobra, busca
bidirecional por substring. `ARQUIVO_PERCENTIL_SCOPUS` é uma planilha de
referência estática -- não depende de nenhuma credencial da API Scopus, por
isso pode ser reaproveitada aqui mesmo numa base sem Scopus vinculado.

> **Nota de fidelidade ao comportamento original:** quando tanto o `issn` da
> base unificada quanto `E-ISSN`/`Print ISSN` (Scopus) são nulos, o
> `pd.merge` do pandas trata os dois nulos como iguais e gera um match "por
> ISSN" mesmo sem nenhum ISSN de fato existir nos dois lados. Esse
> comportamento já existia no notebook original e é preservado aqui.

In [12]:
def formatar_issn(issn):
    """Normaliza um ISSN para 8 dígitos sem hífen, retornando nulo se o valor for vazio/inválido."""
    issn_str = str(issn).replace('-', '').strip()
    if issn_str.lower() in ['nan', 'none', '', 'nat']:
        return pd.NA
    return issn_str.zfill(8)


COLUNAS_PERIODICO_UNIFICADO = [c for c in COLUNAS_PERIODICO if c != 'fonte'] + ['fontes', 'chave_dedup']

print("Carregando e preparando a base completa da Scopus...")
df_scopus_raw = pd.read_excel(ARQUIVO_PERCENTIL_SCOPUS)
df_scopus_raw['Title'] = df_scopus_raw['Title'].astype(str).str.upper().str.strip()
df_scopus_raw['E-ISSN'] = df_scopus_raw['E-ISSN'].apply(formatar_issn)
df_scopus_raw['Print ISSN'] = df_scopus_raw['Print ISSN'].apply(formatar_issn)

mask_comput = df_scopus_raw['Scopus Sub-Subject Area'].str.contains('Comput', case=False, na=False)
titulos_computacao = set(df_scopus_raw[mask_comput]['Title'].unique())
issns_computacao = set(df_scopus_raw[mask_comput]['E-ISSN'].dropna().unique()).union(
                    set(df_scopus_raw[mask_comput]['Print ISSN'].dropna().unique()))

# Um mesmo periódico pode aparecer várias vezes na planilha (uma linha por
# subárea ASJC). Mantemos apenas o percentil mais alto de cada título.
df_scopus_unicos = df_scopus_raw.sort_values(by='Percentile', ascending=False)
df_scopus_unicos = df_scopus_unicos.drop_duplicates(subset=['Title'], keep='first').copy()

colunas_scopus = [
    'Scopus Source ID', 'Title', 'Percentile',
    'Scopus ASJC Code (Sub-subject Area)', 'Scopus Sub-Subject Area', 'E-ISSN', 'Print ISSN'
]
df_scopus_filtro = df_scopus_unicos[colunas_scopus]

print("Preparando a base unificada de periódicos (chave de cruzamento)...")
df_base_periodicos = df_periodicos_unificado.copy()
df_base_periodicos['revista'] = df_base_periodicos['titulo_revista_lattes'].astype(str).str.upper().str.strip()
df_base_periodicos['issn'] = df_base_periodicos['issn'].apply(formatar_issn)

print("Preparação concluída.")

Carregando e preparando a base completa da Scopus...


Preparando a base unificada de periódicos (chave de cruzamento)...
Preparação concluída.


In [13]:
print("Realizando o cruzamento exato (ISSN e Nome Exato)...")

df_match_nome = pd.merge(df_base_periodicos, df_scopus_filtro, left_on='revista', right_on='Title', how='inner')
df_match_e_issn = pd.merge(df_base_periodicos, df_scopus_filtro, left_on='issn', right_on='E-ISSN', how='inner')
df_match_print_issn = pd.merge(df_base_periodicos, df_scopus_filtro, left_on='issn', right_on='Print ISSN', how='inner')

df_sucessos = pd.concat([df_match_nome, df_match_e_issn, df_match_print_issn], ignore_index=True)
df_sucessos = df_sucessos.drop_duplicates(subset=['chave_dedup']).copy()

df_sucessos['Computation Area'] = (
    df_sucessos['Title'].isin(titulos_computacao) |
    df_sucessos['E-ISSN'].isin(issns_computacao) |
    df_sucessos['Print ISSN'].isin(issns_computacao)
)

print(f"Matches exatos encontrados: {len(df_sucessos)}")

Realizando o cruzamento exato (ISSN e Nome Exato)...


Matches exatos encontrados: 1327


In [14]:
print("Busca bidirecional (nome contido) para os artigos restantes...")

chaves_com_match_exato = set(df_sucessos['chave_dedup'])
df_restante = df_base_periodicos[~df_base_periodicos['chave_dedup'].isin(chaves_com_match_exato)].copy()

# Ordena a lista de periódicos Scopus do nome mais longo para o mais curto,
# para evitar que um nome curto "roube" o match de um nome mais específico.
df_scopus_filtro_sorted = df_scopus_filtro.copy()
df_scopus_filtro_sorted['tamanho_titulo'] = df_scopus_filtro_sorted['Title'].str.len()
df_scopus_filtro_sorted = df_scopus_filtro_sorted.sort_values(by='tamanho_titulo', ascending=False)
lista_scopus = df_scopus_filtro_sorted.to_dict('records')


def busca_bidirecional_revista(revista_lattes):
    """Procura, na lista Scopus, um título que contenha (ou esteja contido em) o nome da revista."""
    if pd.isna(revista_lattes) or revista_lattes == 'NAN' or revista_lattes == '':
        return None

    for scopus in lista_scopus:
        titulo_scopus = scopus['Title']
        if pd.notna(titulo_scopus) and titulo_scopus != 'NAN' and titulo_scopus != "":
            if (titulo_scopus in revista_lattes) or (revista_lattes in titulo_scopus):
                return scopus
    return None


resultados_parciais = df_restante['revista'].apply(busca_bidirecional_revista)

mask_encontrados = resultados_parciais.notna()
df_match_parcial = df_restante[mask_encontrados].copy()

if not df_match_parcial.empty:
    dicts_encontrados = resultados_parciais[mask_encontrados]
    for col in colunas_scopus:
        df_match_parcial[col] = [d[col] for d in dicts_encontrados]

    df_match_parcial['Computation Area'] = (
        df_match_parcial['Title'].isin(titulos_computacao) |
        df_match_parcial['E-ISSN'].isin(issns_computacao) |
        df_match_parcial['Print ISSN'].isin(issns_computacao)
    )

    df_sucessos = pd.concat([df_sucessos, df_match_parcial], ignore_index=True)
    df_sucessos = df_sucessos.drop_duplicates(subset=['chave_dedup']).copy()

print(f"Total de sucessos após a busca bidirecional: {len(df_sucessos)}")

Busca bidirecional (nome contido) para os artigos restantes...


Total de sucessos após a busca bidirecional: 1527


In [15]:
print("Isolando e tratando as falhas definitivas...")

df_falhas = df_restante[~mask_encontrados].copy()
df_falhas['Scopus Source ID'] = pd.NA
df_falhas['Title'] = pd.NA
df_falhas['Percentile'] = 0
df_falhas['Scopus ASJC Code (Sub-subject Area)'] = pd.NA
df_falhas['Scopus Sub-Subject Area'] = pd.NA
df_falhas['E-ISSN'] = pd.NA
df_falhas['Print ISSN'] = pd.NA
df_falhas['Computation Area'] = False

print("Consolidando o resultado do cruzamento...")
df_periodicos_tratado = pd.concat([df_sucessos, df_falhas], ignore_index=True)
df_periodicos_tratado['Percentile'] = df_periodicos_tratado['Percentile'].astype(int)
df_periodicos_tratado['Computation Area'] = df_periodicos_tratado['Computation Area'].astype(bool)

# 'match_adequado' indica se o artigo encontrou correspondência válida na Scopus
df_periodicos_tratado['match_adequado'] = df_periodicos_tratado['Scopus Source ID'].notna()

# Sobrescreve os campos de cruzamento com o resultado, mantendo os demais
# campos (id_lattes, titulo_artigo, ano_pub, doi, autores, fontes, chave_dedup) intactos.
df_periodicos_tratado['id_scopus'] = df_periodicos_tratado['Scopus Source ID']
df_periodicos_tratado['titulo_revista_scopus'] = df_periodicos_tratado['Title']
df_periodicos_tratado['maior_percentil'] = df_periodicos_tratado['Percentile']
df_periodicos_tratado['codigo_area_maior_percentil'] = df_periodicos_tratado['Scopus ASJC Code (Sub-subject Area)']
df_periodicos_tratado['area_maior_percentil'] = df_periodicos_tratado['Scopus Sub-Subject Area']
df_periodicos_tratado['issn'] = df_periodicos_tratado['E-ISSN']
df_periodicos_tratado['computation_area'] = df_periodicos_tratado['Computation Area']

df_periodicos_unificado = df_periodicos_tratado[COLUNAS_PERIODICO_UNIFICADO].copy()

print(f"Tabela unificada de periódicos tratada. Total de linhas: {len(df_periodicos_unificado)}")
display(df_periodicos_unificado.sample(min(10, len(df_periodicos_unificado))))

Isolando e tratando as falhas definitivas...
Consolidando o resultado do cruzamento...
Tabela unificada de periódicos tratada. Total de linhas: 1591


,id_lattes,titulo_artigo,titulo_revista_lattes,ano_pub,doi,autores,match_adequado,coautoria_aluno,id_scopus,titulo_revista_scopus,maior_percentil,codigo_area_maior_percentil,area_maior_percentil,issn,computation_area,fontes,chave_dedup
310,8588117212005149,Tuning of 802.11e network parameters,IEEE COMMUNICATIONS LETTERS,2006,http://dx.doi.org/10.1109/LCOMM.2006.1665127,"Freitag, J. ; da Fonseca, N.L.S. ; de Rezende,...",True,NaN,18896,IEEE COMMUNICATIONS LETTERS,93,2611,Modeling and Simulation,NaN,True,LATTES,8588117212005149|DOI:10.1109/lcomm.2006.1665127
259,2704717555047499,Conjecturing the Cognitive Plausibility of an ...,LECTURE NOTES IN COMPUTER SCIENCE,2001,http://dx.doi.org/10.1007/3-540-45720-8_99,"VILELA, I. M. O. ; LIMA, P. M. V.",True,NaN,25674,LECTURE NOTES IN COMPUTER SCIENCE,43,2614,Theoretical Computer Science,16113349,True,LATTES,2704717555047499|DOI:10.1007/3-540-45720-8_99
528,4783565791787812,Adaptative methodology of sustainability indic...,INTERNATIONAL JOURNAL OF GLOBAL ENVIRONMENTAL ...,2009,http://dx.doi.org/10.1504/IJGENVI.2009.027262,"PINHEIRO, Wallace Anacleto ; Barros, Ricardo ;...",True,NaN,23268,INTERNATIONAL JOURNAL OF GLOBAL ENVIRONMENTAL ...,26,3305,"Geography, Planning and Development",17415136,False,LATTES,4783565791787812|DOI:10.1504/ijgenvi.2009.027262
814,8130520066599912,Temporal Profiling for Opportunistic Partnersh...,LECTURE NOTES IN COMPUTER SCIENCE,2008,NaN,"VIVACQUA, A. S. ; Mello, Carlos Eduardo ; de S...",True,NaN,25674,LECTURE NOTES IN COMPUTER SCIENCE,43,2614,Theoretical Computer Science,16113349,True,LATTES,8130520066599912|TIT:TEMPORAL PROFILING FOR OP...
459,3957046121364560,On edge-colouring indifference graphs,THEORETICAL COMPUTER SCIENCE,1997,http://dx.doi.org/10.1016/S0304-3975(96)00264-2,"FIGUEIREDO, C. M. H.; MEIDANIS, J. ; MELLO, C. P.",True,NaN,20571,THEORETICAL COMPUTER SCIENCE,44,2614,Theoretical Computer Science,NaN,True,LATTES,3957046121364560|DOI:10.1016/s0304-3975(96)002...
501,2718664296804955,A Multidimensional Classification Approach for...,IEEE TRANSACTIONS ON BIOMEDICAL ENGINEERING,2008,http://dx.doi.org/10.1109/tbme.2008.915729,"Pedreira, Carlos Eduardo; Costa, Elaine S. ; A...",True,NaN,16318,IEEE TRANSACTIONS ON BIOMEDICAL ENGINEERING,81,2204,Biomedical Engineering,15582531,False,LATTES,2718664296804955|DOI:10.1109/tbme.2008.915729
450,3957046121364560,On Tucker's proof of the Strong Perfect Graph ...,DISCRETE MATHEMATICS,2001,http://dx.doi.org/10.1016/S0012-365X(00)00352-6,"FIGUEIREDO, C. M. H.; GRAVIER, S. ; SALES, C. L.",True,NaN,25892,DISCRETE MATHEMATICS,57,2607,Discrete Mathematics and Combinatorics,NaN,True,LATTES,3957046121364560|DOI:10.1016/s0012-365x(00)003...
96,2002515486942024,Modelling and solving the perfect edge dominat...,OPTIMIZATION LETTERS,2020,http://dx.doi.org/10.1007/s11590-018-1335-x,"DO FORTE, VINICIUS L. ; LIN, MIN CHIH ; LUCENA...",True,NaN,5800228220,OPTIMIZATION LETTERS,69,2606,Control and Optimization,18624480,False,LATTES,2002515486942024|DOI:10.1007/s11590-018-1335-x
1576,7541486051032916,Testes de Integração Aplicados a Software Orie...,PESQUISA NAVAL (SDM),2004,NaN,"Lima, G. M. P. S. ; TRAVASSOS, G. H.",False,NaN,<NA>,<NA>,0,<NA>,<NA>,<NA>,False,LATTES,7541486051032916|TIT:TESTES DE INTEGRACAO APLI...
895,1420784392366957,Deriving scientific workflows from algebraic e...,FUTURE GENERATION COMPUTER SYSTEMS,2017,http://dx.doi.org/10.1016/j.future.2016.08.016,"Marinho, Anderson ; DE OLIVEIRA, DANIEL ; Ogas...",True,NaN,12264,FUTURE GENERATION COMPUTER SYSTEMS,97,1705,Computer Networks and Communications,NaN,True,LATTES,1420784392366957|DOI:10.1016/j.future.2016.08.016


## 8. Cruzamento de Trabalhos de Congresso com a Base de Eventos Classificados

Mesma lógica de match fuzzy (`rapidfuzz`) de `analyse_organizado.ipynb`
(Seção 10): sigla exata isolada por limites de palavra tem prioridade sobre
similaridade textual (`token_set_ratio`) contra o nome do evento em
português e inglês, com corte em 95/100 para evitar falsos positivos.
`ARQUIVO_EVENTOS_CLASSIFICADOS` também é uma base de referência estática.

In [16]:
print("Carregando a base de eventos classificados...")
df_google_raw = pd.read_csv(ARQUIVO_EVENTOS_CLASSIFICADOS)

# A base de eventos usa rótulos B1-B4 para os estratos mais baixos; o projeto
# usa a faixa estendida A1-A8, então B1-B4 são remapeados para A5-A8.
mapeamento_estratos = {'B1': 'A5', 'B2': 'A6', 'B3': 'A7', 'B4': 'A8'}
df_google_raw['Estrato'] = df_google_raw['Estrato'].replace(mapeamento_estratos)
df_google_raw['Nome do evento'] = df_google_raw['Nome do evento'].str.upper()

print("Distribuição de estratos (após remapeamento B1-B4 -> A5-A8):")
display(df_google_raw['Estrato'].value_counts())

Carregando a base de eventos classificados...
Distribuição de estratos (após remapeamento B1-B4 -> A5-A8):


Estrato
A3    171
A4    134
A1    110
A8     90
A2     86
A5     78
A6     60
A7     52
Name: count, dtype: int64

In [17]:
print("Normalizando nomes de eventos (acentos, maiúsculas, espaços) em ambas as bases...")


def limpar_texto(serie):
    """Remove acentos, converte para maiúsculas, colapsa espaços múltiplos e tira espaços nas bordas."""
    return (serie.astype(str)
            .str.normalize('NFKD')
            .str.encode('ascii', errors='ignore')
            .str.decode('utf-8')
            .str.upper()
            .str.replace(r'\s+', ' ', regex=True)
            .str.strip())


df_base_congressos = df_congressos_unificado.copy()
df_base_congressos['evento_limpo'] = limpar_texto(df_base_congressos['titulo_evento_lattes'])
df_google_raw['Nome do evento'] = limpar_texto(df_google_raw['Nome do evento'])
df_google_raw['Nome do evento em inglês'] = limpar_texto(df_google_raw['Nome do evento em inglês'])
df_google_raw['Sigla'] = limpar_texto(df_google_raw['Sigla'])

# Para a busca por substring/sigla funcionar bem nos dois idiomas, ordenamos a
# base de eventos pelo maior nome disponível entre PT e EN, evitando que um
# nome curto "roube" o match de um nome mais longo e específico.
df_google_raw['tamanho_pt'] = df_google_raw['Nome do evento'].str.len()
df_google_raw['tamanho_en'] = df_google_raw['Nome do evento em inglês'].str.len()
df_google_raw['tamanho_max'] = df_google_raw[['tamanho_pt', 'tamanho_en']].max(axis=1)
df_google_raw = df_google_raw.sort_values(by='tamanho_max', ascending=False)

lista_google = df_google_raw.to_dict('records')

print("Normalização concluída.")

Normalizando nomes de eventos (acentos, maiúsculas, espaços) em ambas as bases...
Normalização concluída.


In [18]:
LIMIAR_CORTE_FUZZY = 95  # Escala 0-100; valor alto para evitar falsos positivos


def encontrar_melhor_match_fuzzy(evento_lattes):
    """Procura, na base de eventos classificados, o melhor match fuzzy para um nome de evento.

    Retorna uma tupla: (sigla, nome_do_evento_padronizado, estrato, tipo_match, score_confianca).
    Quando não há match acima do limiar, retorna estrato 'A8' (pior classificação) e tipo 'Sem Match'.
    """
    if pd.isna(evento_lattes) or evento_lattes == 'NAN' or evento_lattes == '':
        return pd.NA, pd.NA, 'A8', 'Sem Match', 0

    melhor_google_match = None
    maior_score_encontrado = 0
    tipo_do_melhor_match = 'Sem Match'

    # --- Tentativa 1: sigla exata isolada por limites de palavra ---
    for google in lista_google:
        sigla = google['Sigla']
        if pd.notna(sigla) and sigla != 'NAN' and sigla != "":
            padrao = r'\b' + re.escape(sigla) + r'\b'
            if re.search(padrao, evento_lattes):
                return google['Sigla'], google['Nome do evento'], google['Estrato'], 'Por Sigla Exata', 100

    # --- Tentativa 2: similaridade fuzzy (token_set_ratio) em PT e EN ---
    for google in lista_google:
        nome_pt = google['Nome do evento']
        nome_en = google['Nome do evento em inglês']

        score_pt = 0
        score_en = 0

        if pd.notna(nome_pt) and nome_pt != 'NAN' and nome_pt != "":
            score_pt = fuzz.token_set_ratio(evento_lattes, nome_pt)

        if pd.notna(nome_en) and nome_en != 'NAN' and nome_en != "":
            score_en = fuzz.token_set_ratio(evento_lattes, nome_en)

        score_atual_max = max(score_pt, score_en)

        if score_atual_max > maior_score_encontrado:
            maior_score_encontrado = score_atual_max
            melhor_google_match = google
            tipo_do_melhor_match = 'Fuzzy Nome PT' if score_pt >= score_en else 'Fuzzy Nome EN'

    # --- Decisão final: o melhor score supera o limiar de segurança? ---
    if maior_score_encontrado >= LIMIAR_CORTE_FUZZY:
        return (
            melhor_google_match['Sigla'],
            melhor_google_match['Nome do evento'],
            melhor_google_match['Estrato'],
            tipo_do_melhor_match,
            maior_score_encontrado
        )

    # Score insuficiente: rejeita o match para evitar falso positivo
    return pd.NA, pd.NA, 'A8', 'Sem Match', maior_score_encontrado

In [19]:
print("Aplicando o match fuzzy a todos os trabalhos de congresso (pode levar alguns segundos)...")

resultados = df_base_congressos['evento_limpo'].apply(encontrar_melhor_match_fuzzy)

df_base_congressos['sigla_evento_google'] = [res[0] for res in resultados]
df_base_congressos['titulo_evento_google'] = [res[1] for res in resultados]
df_base_congressos['estrato'] = [res[2] for res in resultados]
df_base_congressos['tipo_match'] = [res[3] for res in resultados]
df_base_congressos['score_confianca'] = [res[4] for res in resultados]

total_originais = len(df_base_congressos)
qtd_sigla = (df_base_congressos['tipo_match'] == 'Por Sigla Exata').sum()
qtd_fuzzy_pt = (df_base_congressos['tipo_match'] == 'Fuzzy Nome PT').sum()
qtd_fuzzy_en = (df_base_congressos['tipo_match'] == 'Fuzzy Nome EN').sum()
qtd_falhas = (df_base_congressos['tipo_match'] == 'Sem Match').sum()

print("\n--- Relatório de Cruzamento de Eventos ---")
print(f"Total de trabalhos de congresso (unificado): {total_originais}")
if total_originais:
    print(f"Match por Sigla Exata: {qtd_sigla} ({round((qtd_sigla/total_originais)*100, 1)}%)")
    print(f"Match Fuzzy (Nome PT):  {qtd_fuzzy_pt} ({round((qtd_fuzzy_pt/total_originais)*100, 1)}%)")
    print(f"Match Fuzzy (Nome EN):  {qtd_fuzzy_en} ({round((qtd_fuzzy_en/total_originais)*100, 1)}%)")
    print(f"Sem Match:              {qtd_falhas} ({round((qtd_falhas/total_originais)*100, 1)}%)")

    display(
        df_base_congressos[df_base_congressos['tipo_match'] != 'Sem Match']
        [['titulo_evento_lattes', 'titulo_evento_google', 'estrato', 'tipo_match', 'score_confianca']]
        .sample(min(5, total_originais))
    )

Aplicando o match fuzzy a todos os trabalhos de congresso (pode levar alguns segundos)...



--- Relatório de Cruzamento de Eventos ---
Total de trabalhos de congresso (unificado): 3089
Match por Sigla Exata: 1098 (35.5%)
Match Fuzzy (Nome PT):  105 (3.4%)
Match Fuzzy (Nome EN):  672 (21.8%)
Sem Match:              1214 (39.3%)


,titulo_evento_lattes,titulo_evento_google,estrato,tipo_match,score_confianca
2971,E-SCIENCE WORKSHOP,WORKSHOP BRASILEIRO DE E-CIENCIA,A8,Fuzzy Nome EN,100.0
750,3RD INTERNATIONAL WORKSHOP ON SOFTWARE ECOSYST...,WORKSHOP INTERNACIONAL SOBRE ECOSSISTEMAS DE S...,A7,Por Sigla Exata,100.0
1649,IEEE/INNS INTERNATIONAL JOINT CONFERENCE ON NE...,CONFERENCIA INTERNACIONAL CONJUNTA IEEE SOBRE ...,A1,Por Sigla Exata,100.0
2845,SBBD,BRAZILIAN SYMPOSIUM ON DATABASES,A4,Por Sigla Exata,100.0
2028,III CONGRESSO BRASILEIRO DE AUTOMÁTICA,BRAZILIAN CONGRESS OF AUTOMATION,A7,Fuzzy Nome EN,100.0


In [20]:
COLUNAS_CONGRESSO_UNIFICADO = [c for c in COLUNAS_CONGRESSO if c != 'fonte'] + ['fontes', 'chave_dedup']

df_congressos_unificado = df_base_congressos[COLUNAS_CONGRESSO_UNIFICADO].copy()

print("Estrutura final de df_congressos_unificado:")
df_congressos_unificado.info()

Estrutura final de df_congressos_unificado:
<class 'pandas.DataFrame'>
RangeIndex: 3089 entries, 0 to 3088
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   id_lattes             3089 non-null   str   
 1   titulo_artigo         3089 non-null   str   
 2   ano                   3089 non-null   Int64 
 3   doi                   849 non-null    str   
 4   autores               3089 non-null   str   
 5   titulo_evento_lattes  3077 non-null   str   
 6   paginas               2129 non-null   str   
 7   sigla_evento_google   1875 non-null   str   
 8   titulo_evento_google  1875 non-null   str   
 9   estrato               3089 non-null   str   
 10  tipo_match            3089 non-null   str   
 11  coautoria_aluno       0 non-null      object
 12  fontes                3089 non-null   str   
 13  chave_dedup           3089 non-null   object
dtypes: Int64(1), object(2), str(11)
memory usage: 1.2+ MB


## 9. Persistência no DuckDB

Cria (se ainda não existir) e carrega apenas as 4 tabelas exigidas por
`carregar_base_comparacao()` em `app.py`: `tb_professores`,
`tb_artigo_periodico`, `tb_artigo_conferencia` e `tb_orientacoes`, com
exatamente as mesmas colunas dessas tabelas em `analyse_organizado.ipynb`.

In [21]:
print("Conectando ao DuckDB e criando o schema (se ainda não existir)...")
con = duckdb.connect(ARQUIVO_DUCKDB_DESTINO)

# --- Tabela mãe: Professores ---
con.execute("""
CREATE TABLE IF NOT EXISTS tb_professores (
    id_lattes VARCHAR PRIMARY KEY,
    nome_completo VARCHAR,
    nome_citacoes VARCHAR,
    sexo VARCHAR,
    rotulo VARCHAR,
    periodo VARCHAR,
    bolsa_produtividade VARCHAR,
    endereco_profissional VARCHAR,
    atualizacao_cv TIMESTAMP,
    url VARCHAR,
    texto_resumo VARCHAR,
    orcid_id VARCHAR,
    scopus_author_id VARCHAR,
    data_ingresso INTEGER
);
""")

# --- Sequências para os IDs automáticos das tabelas filhas ---
for nome_sequencia in ['seq_id_artigo_periodico', 'seq_id_artigo_conferencia', 'seq_id_orientacao']:
    con.execute(f"CREATE SEQUENCE IF NOT EXISTS {nome_sequencia};")

# --- Tabela Filha: Artigos de Periódico (cruzada com a base de percentil Scopus) ---
con.execute("""
CREATE TABLE IF NOT EXISTS tb_artigo_periodico (
    id_artigo_periodico INTEGER PRIMARY KEY DEFAULT nextval('seq_id_artigo_periodico'),
    id_lattes VARCHAR,
    titulo_artigo VARCHAR NOT NULL,
    titulo_revista_lattes VARCHAR,
    ano_pub INTEGER,
    doi VARCHAR,
    autores VARCHAR,
    match_adequado BOOLEAN,
    coautoria_aluno BOOLEAN,
    id_scopus VARCHAR,
    titulo_revista_scopus VARCHAR,
    maior_percentil INTEGER,
    codigo_area_maior_percentil VARCHAR,
    area_maior_percentil VARCHAR,
    issn VARCHAR,
    computation_area BOOLEAN,
    fontes VARCHAR,
    chave_dedup VARCHAR,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
""")

# --- Tabela Filha: Artigos de Conferência (cruzada com a base de eventos) ---
con.execute("""
CREATE TABLE IF NOT EXISTS tb_artigo_conferencia (
    id_artigo_conferencia INTEGER PRIMARY KEY DEFAULT nextval('seq_id_artigo_conferencia'),
    id_lattes VARCHAR,
    titulo_artigo VARCHAR NOT NULL,
    ano INTEGER,
    doi VARCHAR,
    autores VARCHAR,
    titulo_evento_lattes VARCHAR,
    paginas VARCHAR,
    sigla_evento_google VARCHAR,
    titulo_evento_google VARCHAR,
    estrato VARCHAR,
    tipo_match VARCHAR,
    coautoria_aluno BOOLEAN,
    fontes VARCHAR,
    chave_dedup VARCHAR,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
""")

# --- Tabela Filha: Orientações ---
con.execute("""
CREATE TABLE IF NOT EXISTS tb_orientacoes (
    id_orientacao INTEGER PRIMARY KEY DEFAULT nextval('seq_id_orientacao'),
    id_lattes VARCHAR,
    titulo_trabalho VARCHAR,
    ano_inicio INTEGER,
    orientando VARCHAR,
    tipo_trabalho VARCHAR,
    instituicao VARCHAR,
    curso VARCHAR,
    status VARCHAR,
    nivel VARCHAR,
    ano_conclusao INTEGER,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
""")

# Garante que bancos criados em uma execução anterior deste notebook (antes
# de alguma coluna ter sido adicionada) sejam atualizados ao reexecutar.
for alter_sql in [
    "ALTER TABLE tb_artigo_periodico ADD COLUMN autores VARCHAR",
    "ALTER TABLE tb_artigo_periodico ADD COLUMN doi VARCHAR",
    "ALTER TABLE tb_artigo_periodico ADD COLUMN coautoria_aluno BOOLEAN",
    "ALTER TABLE tb_artigo_periodico ADD COLUMN fontes VARCHAR",
    "ALTER TABLE tb_artigo_periodico ADD COLUMN chave_dedup VARCHAR",
    "ALTER TABLE tb_artigo_conferencia ADD COLUMN coautoria_aluno BOOLEAN",
    "ALTER TABLE tb_artigo_conferencia ADD COLUMN fontes VARCHAR",
    "ALTER TABLE tb_artigo_conferencia ADD COLUMN chave_dedup VARCHAR",
    "ALTER TABLE tb_professores ADD COLUMN orcid_id VARCHAR",
    "ALTER TABLE tb_professores ADD COLUMN scopus_author_id VARCHAR",
    "ALTER TABLE tb_professores ADD COLUMN data_ingresso INTEGER",
]:
    try:
        con.execute(alter_sql)
    except Exception:
        pass  # Coluna já existe -- nada a fazer

print("Schema pronto (4 tabelas, criadas ou já existentes).")

Conectando ao DuckDB e criando o schema (se ainda não existir)...
Schema pronto (4 tabelas, criadas ou já existentes).


In [22]:
print("Limpando dados antigos antes da nova carga (filhas primeiro, mãe depois)...")
for tabela in ['tb_artigo_periodico', 'tb_artigo_conferencia', 'tb_orientacoes']:
    con.execute(f"DELETE FROM {tabela}")
con.execute("DELETE FROM tb_professores")

print("Inserindo os dados tratados...")

# --- Tabela Mãe: Professores ---
if not df_pessoas.empty:
    con.execute("""
        INSERT INTO tb_professores (
            id_lattes, nome_completo, nome_citacoes, sexo, rotulo, periodo,
            bolsa_produtividade, endereco_profissional, atualizacao_cv, url,
            texto_resumo, orcid_id, scopus_author_id, data_ingresso
        )
        SELECT
            id_lattes, nome_completo, nome_citacoes, sexo, rotulo, periodo,
            bolsa_produtividade, endereco_profissional, atualizacao_cv, url,
            texto_resumo, orcid_id, scopus_author_id, data_ingresso
        FROM df_pessoas
    """)

# --- Tabela: Artigos de Periódico ---
if not df_periodicos_unificado.empty:
    con.execute("""
        INSERT INTO tb_artigo_periodico (
            id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub,
            doi, autores, match_adequado, coautoria_aluno, id_scopus,
            titulo_revista_scopus, maior_percentil, codigo_area_maior_percentil,
            area_maior_percentil, issn, computation_area, fontes, chave_dedup
        )
        SELECT
            id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub,
            doi, autores, match_adequado, coautoria_aluno, id_scopus,
            titulo_revista_scopus, maior_percentil, codigo_area_maior_percentil,
            area_maior_percentil, issn, computation_area, fontes, chave_dedup
        FROM df_periodicos_unificado
    """)

# --- Tabela: Artigos de Conferência ---
if not df_congressos_unificado.empty:
    con.execute("""
        INSERT INTO tb_artigo_conferencia (
            id_lattes, titulo_artigo, ano, doi, autores,
            titulo_evento_lattes, paginas, sigla_evento_google,
            titulo_evento_google, estrato, tipo_match, coautoria_aluno, fontes, chave_dedup
        )
        SELECT
            id_lattes, titulo_artigo, ano, doi, autores,
            titulo_evento_lattes, paginas, sigla_evento_google,
            titulo_evento_google, estrato, tipo_match, coautoria_aluno, fontes, chave_dedup
        FROM df_congressos_unificado
    """)

# --- Tabela: Orientações ---
if not df_orientacoes.empty:
    con.execute("""
        INSERT INTO tb_orientacoes (
            id_lattes, titulo_trabalho, ano_inicio, orientando,
            tipo_trabalho, instituicao, curso, status, nivel, ano_conclusao
        )
        SELECT
            id_lattes, titulo_trabalho, ano_inicio, orientando,
            tipo_trabalho, instituicao, curso, status, nivel, ano_conclusao
        FROM df_orientacoes
    """)

con.close()

print(f"Processo finalizado! Banco '{ARQUIVO_DUCKDB_DESTINO}' pronto para ser enviado no uploader "
      f"'Base de Comparação' do módulo 'Comparativo entre Bases' em app.py.")

Limpando dados antigos antes da nova carga (filhas primeiro, mãe depois)...
Inserindo os dados tratados...


Processo finalizado! Banco 'pesquisadores_comparacao.duckdb' pronto para ser enviado no uploader 'Base de Comparação' do módulo 'Comparativo entre Bases' em app.py.


## Verificação final

Reabre o banco gerado em modo somente leitura e roda a mesma checagem de
`carregar_base_comparacao()` em `app.py`, confirmando que as 4 tabelas
obrigatórias existem e mostrando um resumo de volumes.

In [23]:
con_check = duckdb.connect(database=ARQUIVO_DUCKDB_DESTINO, read_only=True)

tabelas_obrigatorias = ['tb_professores', 'tb_artigo_periodico', 'tb_artigo_conferencia', 'tb_orientacoes']
for tabela in tabelas_obrigatorias:
    total = con_check.execute(f"SELECT COUNT(*) FROM {tabela}").fetchone()[0]
    print(f"  {tabela}: {total} linha(s)")

con_check.close()
print("\nOK: todas as tabelas obrigatórias existem. O arquivo pode ser enviado no uploader do app.py.")

  tb_professores: 31 linha(s)
  tb_artigo_periodico: 1591 linha(s)
  tb_artigo_conferencia: 3089 linha(s)
  tb_orientacoes: 2409 linha(s)

OK: todas as tabelas obrigatórias existem. O arquivo pode ser enviado no uploader do app.py.
